# 03 - Experiment Results Analysis & Visualization

This notebook provides a database-driven analysis dashboard for your evolutionary BBOB experiments. It reads directly from the SQLite database (`experiments/results.db`) and incremental checkpoint JSON files (`results/`), loads the data into Pandas DataFrames, performs statistical analysis, and generates interactive plots of the optimization performance.

In [ ]:
# Setup paths and imports
import sys
from pathlib import Path

# Add project root src/ to sys.path regardless of directory name or launch folder
root_dir = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
src_path = str(root_dir / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import os
import glob
import json
import sqlite3
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from storage import get_store

RESULTS_DIR = root_dir / 'results'
DB_PATH = root_dir / 'experiments' / 'results.db'
print("Analysis dashboard environment initialized.")

## 1. Load Data (Checkpoints and Database)
We load data from two potential sources:
1. **Checkpoints (`results/bbob_*_rep_*.json`)**: Incremental run outputs from cluster runs.
2. **SQLite Database (`experiments/results.db`)**: Structured summaries from single or batch notebook executions.

In [ ]:
def load_aggregated_results(results_dir, db_path):
    records = []
    
    # 1. Load checkpoints from results/ folder if available
    pattern = os.path.join(results_dir, "bbob_*_rep_*.json")
    checkpoint_files = glob.glob(pattern)
    for filepath in checkpoint_files:
        try:
            with open(filepath, "r", encoding="utf-8") as f:
                data = json.load(f)
                data_no_code = {k: v for k, v in data.items() if k != "code"}
                records.append({
                    "problem_id": data_no_code.get("problem_id"),
                    "dim": data_no_code.get("dim", 3),
                    "noise_std": data_no_code.get("noise_std", 0.0),
                    "mode": "noisy" if data_no_code.get("noise_std", 0.0) > 0.0 else "clean",
                    "llm_name": data_no_code.get("llm_provider", "unknown"),
                    "best_final_error": data_no_code.get("best_final_error", -data_no_code.get("fitness", float('inf'))),
                    "best_algorithm": data_no_code.get("best_algorithm")
                })
        except Exception as e:
            print(f"Warning: Failed to load {filepath}: {e}")

    # 2. Load from SQLite Database
    if db_path.exists():
        conn = sqlite3.connect(db_path)
        try:
            df_exp = pd.read_sql_query("SELECT * FROM experiments", conn)
            df_iter = pd.read_sql_query("SELECT * FROM iterations", conn)
            conn.close()
            return df_exp, df_iter, pd.DataFrame(records)
        except Exception as e:
            print(f"Error reading database: {e}")
            conn.close()
            
    return pd.DataFrame(), pd.DataFrame(), pd.DataFrame(records)

df_exp, df_iter, df_checkpoints = load_aggregated_results(RESULTS_DIR, DB_PATH)

print(f"Loaded {len(df_exp)} experiment runs from SQLite.")
print(f"Loaded {len(df_iter)} total iterations from SQLite.")
print(f"Loaded {len(df_checkpoints)} checkpoint records from results/.")

## 2. Print Experiment Results Table
We present a formatted tabular report of all completed experiments in the SQLite database.

In [ ]:
if not df_exp.empty:
    report_df = df_exp[[
        'problem_id', 'dim', 'mode', 'llm_name', 'best_iteration', 'best_algorithm', 'best_final_error', 'created_at'
    ]].copy()
    report_df.columns = [
        'Problem ID', 'Dim', 'Mode', 'LLM Provider', 'Best Iteration', 'Best Algorithm', 'Best Final Error', 'Run Date'
    ]
    report_df = report_df.sort_values(by=['Problem ID', 'Dim', 'Mode', 'LLM Provider'])
    
    # Format error column
    report_df['Best Final Error'] = report_df['Best Final Error'].apply(
        lambda x: f"{x:.6e}" if isinstance(x, (int, float)) and not np.isnan(x) else "N/A"
    )
    
    print("=== Completed BBOB Optimization Experiments ===")
    display(report_df)
else:
    print("No experiment runs available in SQLite database.")

## 3. Thesis Performance Statistics
Compute aggregate statistics across multiple runs, grouping by problem ID, dimension, mode, and LLM provider.

In [ ]:
if not df_exp.empty:
    # Performance Summary Stats per Problem Configuration
    stats_df = df_exp.groupby(['problem_id', 'dim', 'mode', 'llm_name'])['best_final_error'].agg([
        ('Runs Count', 'count'),
        ('Min Error', 'min'),
        ('Max Error', 'max'),
        ('Mean Error', 'mean'),
        ('Std Dev', 'std')
    ]).reset_index()
    
    stats_df.rename(columns={
        'problem_id': 'Problem ID',
        'dim': 'Dim',
        'mode': 'Mode',
        'llm_name': 'LLM Target'
    }, inplace=True)
    
    print("--- Summary Statistics of Best Discovered Algorithms ---")
    display(stats_df.round(6))
    
    # Execution Efficiency Summary
    if not df_iter.empty and 'runtime_seconds' in df_iter.columns:
        df_merged = pd.merge(df_exp, df_iter, left_on='id', right_on='experiment_id', suffixes=('_exp', '_iter'))
        speed_df = df_merged.groupby(['problem_id_exp', 'dim_exp', 'mode_exp', 'llm_name_exp']).agg({
            'runtime_seconds': ['mean', 'sum'],
            'llm_generation_time': 'mean',
            'converged': lambda x: f"{(x.sum() / len(x)) * 100:.1f}%"
        }).reset_index()
        speed_df.columns = [
            'Problem ID', 'Dim', 'Mode', 'LLM Target',
            'Avg Iteration Time (s)', 'Total Runtime (s)', 'Avg LLM Gen Time (s)', 'Convergence Rate'
        ]
        print("\n--- Execution Stats & Convergence Summary ---")
        display(speed_df.round(2))
else:
    print("No database records available to calculate statistics.")

## 4. Visualizations & Plots
Generate interactive charts to explore convergence performance, error distributions, and runtimes.

In [ ]:
if not df_iter.empty and not df_exp.empty:
    df_merged = pd.merge(df_exp, df_iter, left_on='id', right_on='experiment_id', suffixes=('_exp', '_iter'))
    
    # Plot 1: Convergence trajectories across iterations
    fig_conv = go.Figure()
    for exp_id in df_merged['experiment_id'].unique():
        exp_subset = df_merged[df_merged['experiment_id'] == exp_id].sort_values(by='iteration')
        meta_row = exp_subset.iloc[0]
        label = f"BBOB-{meta_row['problem_id_exp']} (Dim={meta_row['dim_exp']}, Mode={meta_row['mode_exp']}, {meta_row['llm_name_exp']})"
        
        fig_conv.add_trace(go.Scatter(
            x=exp_subset['iteration'],
            y=exp_subset['final_error'],
            mode='lines+markers',
            name=label,
            hovertext=exp_subset['algorithm_name']
        ))
        
    fig_conv.update_layout(
        title='Evolutionary Convergence Trajectories (Final Error vs. Iteration)',
        xaxis_title='Evolutionary Iteration',
        yaxis_title='Final Error (Log Scale)',
        yaxis_type='log',
        template='plotly_white',
        height=600,
        width=1000
    )
    fig_conv.show()
    
    # Plot 2: Boxplot comparing final errors across different LLM providers/modes
    fig_box = px.box(
        df_exp,
        x='llm_name',
        y='best_final_error',
        color='mode',
        points='all',
        title='Discovered Algorithms: Best Final Error Distribution',
        labels={'llm_name': 'LLM Target Provider', 'best_final_error': 'Best Final Error', 'mode': 'Optimization Mode'}
    )
    fig_box.update_layout(
        template='plotly_white',
        yaxis_type='log',
        height=500,
        width=800
    )
    fig_box.show()
    
    # Plot 3: Average Runtime distribution
    fig_runtime = px.bar(
        df_merged.groupby(['problem_id_exp', 'llm_name_exp'])['runtime_seconds'].mean().reset_index(),
        x='problem_id_exp',
        y='runtime_seconds',
        color='llm_name_exp',
        barmode='group',
        title='Average Execution Runtime per Iteration by BBOB Problem and LLM Target',
        labels={'problem_id_exp': 'BBOB Problem ID', 'runtime_seconds': 'Avg Runtime (seconds)', 'llm_name_exp': 'LLM Target'}
    )
    fig_runtime.update_layout(
        template='plotly_white',
        height=450,
        width=800
    )
    fig_runtime.show()
else:
    print("No database records available to plot charts.")